# Interfacing: API, Frontend and MCP

In the previous notebook we made a language model return JSON that follows a schema. This notebook is about what happens next: **how does that JSON travel between programs?**

We follow one schema from start to finish, the **product** of the Auto Quote catalogue:

1. Remember JSON? The schema that will be our red line
2. What is an API? A tiny API, built and called from this notebook
3. Security: why it belongs in the backend
4. Frontend: just another client of the same API
5. AI agents: documentation is for humans, MCP is for models

Everything runs inside this notebook: the servers start in the background, and we call them from the next cell.

In [ ]:
# Setup - Can be ignored
import functools
import json
import os
import secrets
import threading
import time
from pathlib import Path
from typing import Annotated
from uuid import uuid4

import requests
import uvicorn
from dotenv import load_dotenv
from fastapi import APIRouter, Depends, FastAPI, Header, HTTPException
from openai import OpenAI
from pydantic import BaseModel, Field

MODEL = "gpt-6-luna"  # text and tool calling

# walk up from the working directory to the repo root (the folder with .git)
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())


@functools.cache
def get_client() -> OpenAI:
    load_dotenv(REPO_ROOT / ".env.local")
    return OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL") or None)


client = get_client()

RUNNING_SERVERS: dict[int, tuple[uvicorn.Server, threading.Thread]] = {}


def serve(app, port: int) -> str:
    """Starts an ASGI app in a background thread and returns its base URL. Running a cell twice restarts the server."""
    stop(port)
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    while not server.started:
        time.sleep(0.05)
    RUNNING_SERVERS[port] = (server, thread)
    return f"http://127.0.0.1:{port}"


def stop(port: int) -> None:
    if port in RUNNING_SERVERS:
        server, thread = RUNNING_SERVERS.pop(port)
        server.should_exit = True
        thread.join()


def show(response: requests.Response) -> None:
    print(response.status_code, response.reason)
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))

## 1. Remember JSON?

In the previous notebook, a JSON Schema described exactly which keys and types we expect from the model. JSON is the common language of our application, and you can see it in two places:

- **At rest**: the "database" of the prototype is a set of JSON files in `backend/data/` (for example `products.json`, created the first time the backend runs).
- **In transit**: every message between the frontend and the backend is a JSON payload. This is what this notebook is about.

Our red line is the product of the catalogue. It is a trimmed copy of `ProductInput` and `Product` from `backend/src/auto_quote/models/product.py`. We imagine a metallurgy company that sells profiles, sheets and bars, and machining services. The data below is mock data.

In [ ]:
class ProductInput(BaseModel):
    product_reference: str = Field(min_length=1)
    description: str
    unit: str
    unit_price: float | None = Field(ge=0, description="Price in CHF per unit, null when the price is unknown.")
    spec: dict[str, str]


class Product(ProductInput):
    id: str
    create_timestamp: int
    update_timestamp: int


CATALOGUE = [
    {
        "product_reference": "HEA-200",
        "description": "Structural steel beam HEA 200",
        "unit": "m",
        "unit_price": 78.0,
        "spec": {"steel_grade": "S355JR", "weight": "42.3 kg/m"},
    },
    {
        "product_reference": "SS-304-2",
        "description": "Stainless steel sheet 2 mm",
        "unit": "m²",
        "unit_price": 89.0,
        "spec": {"alloy": "1.4301 (AISI 304)", "thickness": "2 mm", "finish": "2B"},
    },
    {
        "product_reference": "AL-6060-R40",
        "description": "Aluminium round bar, diameter 40 mm",
        "unit": "m",
        "unit_price": 38.5,
        "spec": {"alloy": "EN AW-6060", "diameter": "40 mm"},
    },
    {
        "product_reference": "CUT-LASER",
        "description": "Laser cutting of steel sheets",
        "unit": "hour",
        "unit_price": 145.0,
        "spec": {"max_thickness": "20 mm"},
    },
    {
        "product_reference": "WELD-MIG",
        "description": "MIG/MAG welding by a certified welder",
        "unit": "hour",
        "unit_price": 98.0,
        "spec": {"certification": "ISO 9606-1"},
    },
]

print(json.dumps(CATALOGUE[0], indent=2))

The same product, as the JSON Schema that Pydantic generates. This is exactly the kind of schema we gave to the model in the previous notebook. Here it describes what our own API accepts.

In [ ]:
print(json.dumps(ProductInput.model_json_schema(), indent=2))

## 2. What is an API?

An **API** (Application Programming Interface) is a door into a program that other programs can use. Instead of a person clicking on a screen, a program sends a request and receives a response.

```
 client                                   server
(frontend, CRM, script, AI agent)        (our backend)
        │                                     │
        │  POST /api/v1/products + JSON body  │
        │ ──────────────────────────────────► │  validates, stores
        │                                     │
        │  200 OK  + JSON response            │
        │ ◄────────────────────────────────── │
```

Most web APIs (REST) work over HTTP: the **client** sends a request, the **server** sends back a response. A request has a **method** (what to do), a **path** (on what), headers, and optionally a **JSON body**. A response has a **status code**, headers and usually a JSON body.

The client can be our frontend, a CRM, a script or an AI agent. It does not matter to the server.

| Method | Meaning | Example |
|---|---|---|
| `GET` | read | `GET /api/v1/products` |
| `POST` | create | `POST /api/v1/products` with a product as body |
| `PUT` | replace | `PUT /api/v1/products/{product_id}` with a product as body |
| `DELETE` | remove | `DELETE /api/v1/products/{product_id}` |

| Status | Meaning |
|---|---|
| `200` | worked |
| `401` | you are not identified |
| `404` | it does not exist |
| `422` | the payload does not match the schema |

Because every language can send HTTP and parse JSON, an API connects very different things. This is why the JSON schema matters: it is the contract both sides agree on.

### A tiny API

With [FastAPI](https://fastapi.tiangolo.com/), the Pydantic models from above become the contract: a payload that does not match `ProductInput` never reaches our code. We build the same three product endpoints as the real app (`backend/src/auto_quote/routes/product.py`): list, create and update, under the same `/api/v1/products` path. The store is a Python dict that lives as long as the notebook, whereas Auto Quote writes to JSON files.


In [ ]:
PRODUCTS_PATH = "/api/v1/products"
PRODUCT_PATH = f"{PRODUCTS_PATH}/{{product_id}}"

products: dict[str, Product] = {}
router = APIRouter()


def now_timestamp() -> int:
    return time.time_ns() // 1_000_000


@router.get(PRODUCTS_PATH)
def get_products() -> list[Product]:
    return list(products.values())


@router.post(PRODUCTS_PATH)
def create_product(product_input: ProductInput) -> Product:
    timestamp = now_timestamp()
    product = Product(id=str(uuid4()), create_timestamp=timestamp, update_timestamp=timestamp, **product_input.model_dump())
    products[product.id] = product
    return product


@router.put(PRODUCT_PATH)
def update_product(product_id: str, product_input: ProductInput) -> Product:
    if product_id not in products:
        raise HTTPException(status_code=404, detail=f"Product {product_id} not found")
    product = products[product_id].model_copy(update={**product_input.model_dump(), "update_timestamp": now_timestamp()})
    products[product_id] = product
    return product


app = FastAPI(title="Products")
app.include_router(router)

BASE_URL = serve(app, port=8001)
print("API running on", BASE_URL)


The cell finished, but the server did not stop: it runs in a **background thread** of this notebook and waits for requests. This means you can simply run the next cells, which call it.

- Running the cell above again restarts the server, with an empty catalogue.
- To stop it: run `stop(8001)` in a cell, or run the last cell of the notebook, which stops every server started here. Restarting or closing the kernel stops them as well.

Now we are the client. First, an empty catalogue:


In [ ]:
show(requests.get(f"{BASE_URL}{PRODUCTS_PATH}"))

We send our catalogue, one JSON payload per product. `json=` serialises the dict and sets the `Content-Type: application/json` header.

In [ ]:
for product_payload in CATALOGUE:
    response = requests.post(f"{BASE_URL}{PRODUCTS_PATH}", json=product_payload)
    print(response.status_code, response.json()["id"], response.json()["product_reference"])

`PUT` replaces a product: we send the complete product, with a new price, to the URL of its `id`. Note that `update_timestamp` changes and `create_timestamp` does not.

In [ ]:
hea_200 = next(product for product in requests.get(f"{BASE_URL}{PRODUCTS_PATH}").json() if product["product_reference"] == "HEA-200")
price_update = {**CATALOGUE[0], "unit_price": 82.0}
show(requests.put(f"{BASE_URL}{PRODUCTS_PATH}/{hea_200['id']}", json=price_update))

Updating a product that does not exist gives a `404`. In Auto Quote this is the `NotFoundError` raised by the controller, translated into a status code at the edge (`backend/src/auto_quote/server.py`).

In [ ]:
show(requests.put(f"{BASE_URL}{PRODUCTS_PATH}/DOES-NOT-EXIST", json=CATALOGUE[0]))

A payload that breaks the schema is refused before our code runs. Here the price is negative and `spec` is missing. The answer says exactly what is wrong, which is useful for a developer and also for an AI agent.

In [ ]:
bad_payload = {"product_reference": "XX-1", "description": "Broken", "unit": "piece", "unit_price": -5}
show(requests.post(f"{BASE_URL}{PRODUCTS_PATH}", json=bad_payload))

## 3. Security

So far, anyone who can reach the server can read and write our catalogue. The goal of this section is not to cover security, which is a topic of its own, but to make one point:

> **Security has to live in the backend.**

A frontend can hide a button or a page, but that only helps honest users. The API is reachable without the frontend, exactly as we just did from Python. Whoever wants to misuse it will not use our interface. So every endpoint has to check **who is calling** and **what they may do**, on the server.

Below, the simplest form: the caller must send a secret key in a header. We reuse the same router and the same data and only put a check in front of every endpoint.

In [ ]:
API_KEY = "demo-key"  # demo only: a real key is read from the environment and never written in code


def require_api_key(x_api_key: Annotated[str | None, Header()] = None) -> None:
    if x_api_key is None or not secrets.compare_digest(x_api_key.encode(), API_KEY.encode()):
        raise HTTPException(status_code=401, detail="Missing or invalid API key")


secured_app = FastAPI(title="Products (secured)")
secured_app.include_router(router, dependencies=[Depends(require_api_key)])

SECURED_URL = serve(secured_app, port=8002)
print("Secured API running on", SECURED_URL)

In [ ]:
print("No key:")
show(requests.get(f"{SECURED_URL}{PRODUCTS_PATH}"))

print("\nWrong key:")
show(requests.get(f"{SECURED_URL}{PRODUCTS_PATH}", headers={"X-API-Key": "guess"}))

In [ ]:
API_HEADERS = {"X-API-Key": API_KEY}
response = requests.get(f"{SECURED_URL}{PRODUCTS_PATH}", headers=API_HEADERS)
print(response.status_code, [p["product_reference"] for p in response.json()])

In Auto Quote every route declares the user it works for, and FastAPI runs the check before the route body. From `backend/src/auto_quote/routes/product.py`:

```python
CurrentUser = Annotated[User, Depends(auth_provider.get_current_user)]

@router.get(ApiPaths.PRODUCTS)
async def get_products(user: CurrentUser) -> list[Product]:
    return await product_controller.get_products()
```

It is the same mechanism as our `Depends(require_api_key)`. The prototype identifies users with a session cookie (`auth/session_cookie.py`); a production setup would plug in a real login behind the same `AuthProvider`.

Three habits that follow from this:

- Check identity and permissions on the server, on every endpoint.
- Never put a secret in the frontend or in a notebook you share: anything the browser receives, the user can read.
- Validate the input on the server. The schema above is that validation.

## 4. Frontend

The frontend of Auto Quote (React, in `frontend/`) is **a client of the API**, exactly like our `requests` calls. It has no direct access to the data: when you open the catalogue page, the browser sends `GET /api/v1/products` and draws the JSON it gets back. When you save a product, it sends a `POST` with a JSON payload.

Here is the file that does this for products. It has the same three calls as our tiny API, and the TypeScript `ProductInput` interface mirrors our Pydantic model: it is the same contract, written twice.

In [ ]:
print((REPO_ROOT / "frontend" / "src" / "services" / "product.ts").read_text())

Our calls above, in comparison:

| | Frontend (TypeScript) | This notebook (Python) |
|---|---|---|
| list | `this.get<Product[]>(PRODUCTS_PATH)` | `requests.get(f"{BASE_URL}{PRODUCTS_PATH}")` |
| create | `this.post<Product>(PRODUCTS_PATH, productInput)` | `requests.post(f"{BASE_URL}{PRODUCTS_PATH}", json=payload)` |
| update | ``this.put<Product>(`${PRODUCTS_PATH}/${productId}`, productInput)`` | `requests.put(f"{BASE_URL}{PRODUCTS_PATH}/{product_id}", json=payload)` |

You can watch this live: run the app, open the browser developer tools (**F12**), go to the **Network** tab and use the catalogue page. Every click shows a request with its JSON payload and the JSON response.

Two consequences for the design:

- The API is the product, the screen is one way to use it. A second client (a CRM integration, a script, an agent) does not need a second backend.
- The frontend can validate the form for comfort, but only the backend decides, as we saw in the security section.

## 5. Allowing the AI to use our API

### Documentation: made for humans

How does a developer learn what an API offers? They read its documentation, and FastAPI generates it from our code. Open <http://127.0.0.1:8001/docs> in your browser (the tiny API from section 2 is still running): it is an interactive page where you can read every endpoint and its payload, and try it out. For Auto Quote, the same page is at <http://localhost:8000/docs> when the backend runs.

Behind that page there is a machine-readable file, the **OpenAPI** document. It is JSON, and the schemas inside are the JSON Schemas we saw above.


In [ ]:
openapi = requests.get(f"{BASE_URL}/openapi.json").json()
print(list(openapi["paths"]))
print(json.dumps(openapi["components"]["schemas"]["ProductInput"], indent=2))

The OpenAPI document is complete, but it is written for a person who then writes the code: someone decides which call answers which need, and handles the URLs, the headers and the errors. An AI model cannot open a documentation page and click on *Try it out*. It needs something else: a list of actions it can request, each with a name, a plain-language description and an input schema, in a standard format that every agent understands.


### MCP: the same door, made for models

Can an AI agent use our API? Yes, with the right door. **MCP** (Model Context Protocol) is an open standard for exactly that. An MCP server exposes a list of **tools**. Each tool has:

- a **name**,
- a **description** in plain language (this is what the model reads to decide when to use it),
- an **input schema**, a JSON Schema again.

```
 agent (LLM + MCP client)               MCP server                      our API
        │  list_tools                       │                              │
        │ ────────────────────────────────► │                              │
        │ ◄──── names, descriptions, schemas│                              │
        │  call_tool get_product {...}      │  GET /products/HEA-200 + key │
        │ ───────────────────────────────────────────────────────────────► │
        │ ◄──────────────── result ─────────────────────────────────────── │
```

The MCP server is a thin layer in front of the API we already have. The API key stays in the MCP server: the agent never sees it.

### A tiny MCP server

We use [FastMCP](https://gofastmcp.com), which is a dependency of the backend. We expose two read-only tools. The docstrings are not decoration: they are the instructions the model reads.

In [ ]:
from fastmcp import Client, FastMCP

mcp_server = FastMCP("metal-catalogue")


def fetch_products() -> list[dict]:
    response = requests.get(f"{SECURED_URL}{PRODUCTS_PATH}", headers=API_HEADERS)
    response.raise_for_status()
    return response.json()


@mcp_server.tool
def list_products() -> list[dict]:
    """List every product of the catalogue with its id, reference, description, unit, unit price in CHF and specifications."""
    return fetch_products()


@mcp_server.tool
def get_product(product_reference: str) -> dict:
    """Get one product of the catalogue by its reference, for example HEA-200."""
    # The API has no lookup by reference: a tool can be shaped around a task instead of around an endpoint.
    for product in fetch_products():
        if product["product_reference"] == product_reference:
            return product
    raise ValueError(f"No product with reference {product_reference}")


MCP_URL = serve(mcp_server.http_app(), port=8003) + "/mcp"
print("MCP server running on", MCP_URL)


Now we are an MCP client. The first thing a client does is ask which tools exist. Look at `input_schema`: it is the same kind of JSON Schema as everywhere else in this notebook.

In [ ]:
async def list_mcp_tools():
    async with Client(MCP_URL) as mcp_client:
        return await mcp_client.list_tools()


for tool in await list_mcp_tools():
    print(tool.name)
    print(" ", tool.description)
    print(" ", json.dumps(tool.input_schema))

### Connecting the server to an agent

An agent is a loop around the model:

1. Send the question to the model together with the tool descriptions.
2. If the model answers with a **tool call**, run it on the MCP server, send the result back and go to 1.
3. If the model answers with text, we are done.

The model never calls anything itself: it only asks for a call, and our code decides whether to run it. The tool schemas from MCP go directly into the `tools` parameter, because both sides speak JSON Schema.

In [ ]:
def to_openai_tool(tool) -> dict:
    return {"type": "function", "name": tool.name, "description": tool.description, "parameters": tool.input_schema}


async def ask_agent(question: str, max_steps: int = 5) -> str:
    async with Client(MCP_URL) as mcp_client:
        tools = [to_openai_tool(tool) for tool in await mcp_client.list_tools()]
        messages = [{"role": "user", "content": question}]

        for _ in range(max_steps):
            response = client.responses.create(model=MODEL, input=messages, tools=tools)
            messages += response.output
            tool_calls = [item for item in response.output if item.type == "function_call"]
            if not tool_calls:
                return response.output_text

            for call in tool_calls:
                arguments = json.loads(call.arguments)
                print(f"tool call: {call.name}({arguments})")
                result = await mcp_client.call_tool(call.name, arguments, raise_on_error=False)
                output = "\n".join(block.text for block in result.content)
                messages.append({"type": "function_call_output", "call_id": call.call_id, "output": output})

    raise RuntimeError(f"No answer after {max_steps} steps")

In [ ]:
print(await ask_agent("How much does one metre of HEA-200 cost and which steel grade is it?"))

In [ ]:
print(await ask_agent("Which of our products are sold per hour, and what would 6 hours of welding cost?"))

Look at the printed tool calls: the model picked the tool and filled in the arguments from the schema, and our API answered with the same JSON as before. We wrote no code that links a question to an endpoint.

The same MCP server works with any MCP client, not only with our loop. With the notebook running, Claude Code can connect to it with:

```sh
claude mcp add --transport http metal-catalogue http://127.0.0.1:8003/mcp
```

Two things to keep in mind when giving an agent access to your API:

- **Start read-only.** Every tool you add is something the agent can do on its own. A tool that writes or deletes deserves a lot more care than the two above.
- **The tools are your security boundary.** The agent can only do what the MCP server exposes, and the API still checks the key on every request.

## Takeaways

- JSON is what travels between our programs, and a JSON Schema is the contract both sides agree on.
- An API lets any program (a frontend, a CRM, a script) use our backend through requests and JSON payloads. FastAPI turns a Pydantic model into validation and documentation.
- Security belongs in the backend, on every endpoint. Hiding something in the frontend is not security.
- The frontend is one client of the API; it does not need its own access to the data.
- Documentation is for humans. MCP lets an AI agent discover and call our API: tools with a name, a description and an input schema. It is a thin layer over the API we already have.

### Where to see and change each part in Auto Quote

Start the app (`make dev` in `backend/`, `npm run dev` in `frontend/`) and look at it from each angle. File links are relative to this notebook.

| Part | Observe it | Modify it |
|---|---|---|
| **JSON and schema** | The catalogue is stored in `backend/data/products.json` (created on first run). The product schema is [`models/product.py`](../src/auto_quote/models/product.py). | In the app, [Admin → Documents](http://localhost:3000/admin/document-retrieval), table *Products*: add or edit a product and watch `products.json` change. To add a field, change `ProductInput` in `models/product.py` and the matching interface in [`services/product.ts`](../../frontend/src/services/product.ts). |
| **API** | [http://localhost:8000/docs](http://localhost:8000/docs): every route of the app, with *Try it out*. The product routes are in [`routes/product.py`](../src/auto_quote/routes/product.py), the paths in [`routes/paths.py`](../src/auto_quote/routes/paths.py). | Add a route to `routes/product.py` (for example a `GET` by id, which the app does not have) and the controller method behind it in [`controllers/product.py`](../src/auto_quote/controllers/product.py). |
| **Security** | In the browser developer tools (**F12**), *Application* tab, the cookie `auto_quote_session` identifies the caller. The check is [`auth/session_cookie.py`](../src/auto_quote/auth/session_cookie.py), behind the interface in [`auth/auth_provider.py`](../src/auto_quote/auth/auth_provider.py). | In [`main.py`](../src/auto_quote/main.py), the line `auth_provider = SessionCookieProvider(...)` is the only place that chooses the implementation. A new provider, for example an API key like in this notebook, is plugged in there. |
| **Frontend** | **F12**, *Network* tab, then open [Admin → Documents](http://localhost:3000/admin/document-retrieval) and save a product: you see the `PUT /api/v1/products/{id}` request with its JSON payload. | The calls are in [`services/product.ts`](../../frontend/src/services/product.ts). The table and the edit dialog are [`products-table.tsx`](../../frontend/src/components/admin/products-table.tsx) and [`product-dialog.tsx`](../../frontend/src/components/admin/product-dialog.tsx). |
| **MCP** | There is no MCP server in the app yet. The tiny one above shows what it looks like, and `prototyping_21_api_to_mcp_optional_deep_dive.ipynb` converts the real app. | The tool names and descriptions are what the model reads: change a docstring in the notebook and ask the agent the same question again. |


## Shutting down

This stops the three servers started in this notebook (ports 8001, 8002 and 8003). To stop only one, run `stop(8001)` for example.

In [ ]:
# Stops the three servers started in this notebook
for port in list(RUNNING_SERVERS):
    stop(port)